# MOMA

## 1a. Process Optimization output

This code creates a reference set from the different seeds

NOTE: For MMBorg archives, run the script to convert it to the format recognized by older code with ema-workbench.

In [1]:
!python justice/util/borg_archive_processor.py \
    --archive /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/mm_intermediate_5.zip \
    --base-name MOMA_200000_ref2 \
    --step 10000 \
    --island-offset 0

!python justice/util/borg_archive_processor.py \
    --archive /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/mm_intermediate_4.zip \
    --base-name MOMA_200000_ref2 \
    --step 10000 \
    --island-offset 5


Archive        : /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/mm_intermediate_5.zip
Islands found  : 5
Island offset  : 0  → indices 0..4
Step filter    : every 10000
Output dir     : /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION

[OK] /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_200000_ref2_0.tar.gz  (source island: mm_0)
[OK] /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_200000_ref2_1.tar.gz  (source island: mm_1)
[OK] /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_200000_ref2_2.tar.gz  (source island: mm_2)
[OK] /Users/palokbiswas/Desktop/pollockdevis_git/JUSTICE/data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_200000_ref2_3.tar.gz  (source island: mm_3)
[OK] /Users/palokbiswas/Desktop/polloc

## 1b. Create reference set

In [6]:
# ── Section 1b: Compute Reference Set + Epsilon Filtering ────────────────────
from solvers.convergence.pareto import eps_sort
import numpy as np
import pandas as pd
import zipfile
import tarfile
import io
import os
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
base_path  = "data/temporary/MOMA_DATA/200k"
joint_path = f"{base_path}/JOINT_OPTIMIZATION"

zip_5_islands = f"{joint_path}/mm_intermediate_5.zip"
zip_4_islands = f"{joint_path}/mm_intermediate_4.zip"

list_of_objectives = [
    "macro_welfare_R5ASIA",
    "macro_welfare_R5LAM",
    "macro_welfare_R5MAF",
    "macro_welfare_R5OECD",
    "macro_welfare_R5REF",
    "fraction_above_threshold",
]

maximize_cols = [1, 2, 3, 4, 5]

# Two filtering levels:
#   fine  → large reference set, used for hypervolume calculation
#   coarse → small policy set, used for simulation & analysis
epsilon_configs = {
    "fine":   {"epsilons": [0.001, 0.001, 0.001, 0.001, 0.001, 0.001],
               "filename": "MOMA_reference_set_fine.csv"},
    "coarse": {"epsilons": [50, 50, 50, 50, 50, 0.1],
               "filename": "COMBINED_MOMA_epsilon_nondominated_set.csv"},
}

# ── HELPER: extract 200000.csv from nested tar.gz inside a zip ────────────────
def extract_final_csv_from_zip(zip_path: str) -> pd.DataFrame:
    """
    Opens zip_path, finds the nested MOMADPS_200000_*.tar.gz,
    extracts tmp/200000.csv from it, returns as DataFrame.
    """
    dfs = []
    with zipfile.ZipFile(zip_path, "r") as outer_zip:
        tar_names = [n for n in outer_zip.namelist() if n.endswith(".tar.gz")]
        print(f"      Found {len(tar_names)} tar.gz files in {os.path.basename(zip_path)}")

        for tar_name in tar_names:
            with outer_zip.open(tar_name) as tar_bytes:
                with tarfile.open(fileobj=io.BytesIO(tar_bytes.read()), mode="r:gz") as tar:
                    csv_members = [m for m in tar.getmembers()
                                   if m.name.endswith("200000.csv")]
                    if not csv_members:
                        print(f"      [WARN] No 200000.csv found in {tar_name}, skipping.")
                        continue
                    f = tar.extractfile(csv_members[0])
                    df = pd.read_csv(f)
                    dfs.append(df)
                    print(f"      Loaded {len(df)} rows from {tar_name} → {csv_members[0].name}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ── STEP 1: Extract and concatenate 200000.csv from both zips ────────────────
print("[1/3] Extracting final archives from both zip files...")

print(f"\n  Processing: {zip_5_islands}")
df_5 = extract_final_csv_from_zip(zip_5_islands)

print(f"\n  Processing: {zip_4_islands}")
df_4 = extract_final_csv_from_zip(zip_4_islands)

df_combined = pd.concat([df_5, df_4], ignore_index=True)
combined_path = f"{joint_path}/MOMA_combined_reference_set_9_islands.csv"
df_combined.to_csv(combined_path, index=False)

print(f"\n  Rows from 5-island run : {len(df_5)}")
print(f"  Rows from 4-island run : {len(df_4)}")
print(f"  Total combined rows    : {len(df_combined)}")
print(f"  Saved to               : {combined_path}\n")

# ── STEP 2: Filter at both epsilon levels ─────────────────────────────────────
print("[2/3] Applying epsilon-nondominated filtering...")

table = df_combined.reset_index()[["index"] + list_of_objectives].to_numpy()

for level, cfg in epsilon_configs.items():
    tagalongs = eps_sort(
        table,
        objectives=[1, 2, 3, 4, 5, 6],
        epsilons=cfg["epsilons"],
        maximize=maximize_cols,
    )
    tagalongs       = np.array(tagalongs)
    tagalongs[:, 0] = tagalongs[:, 0].astype(int)
    pareto_idx      = np.unique(tagalongs[:, 0])
    filtered_df     = df_combined.loc[pareto_idx].copy()

    out_file = f"{joint_path}/{cfg['filename']}"
    filtered_df.to_csv(out_file, index=False)
    print(f"  [{level:6s}]  epsilons: {cfg['epsilons']}"
          f"  →  {len(filtered_df):4d} policies  →  {out_file}")

print("\n[3/3] Summary of outputs:")
print(f"  Combined raw set  : {combined_path}")
for level, cfg in epsilon_configs.items():
    print(f"  {level:6s} filtered : {joint_path}/{cfg['filename']}")


[1/3] Extracting final archives from both zip files...

  Processing: data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/mm_intermediate_5.zip
      Found 1 tar.gz files in mm_intermediate_5.zip
      Loaded 264 rows from MOMADPS_200000_66.tar.gz → tmp/200000.csv

  Processing: data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/mm_intermediate_4.zip
      Found 1 tar.gz files in mm_intermediate_4.zip
      Loaded 1016 rows from MOMADPS_200000_555.tar.gz → tmp/200000.csv

  Rows from 5-island run : 264
  Rows from 4-island run : 1016
  Total combined rows    : 1280
  Saved to               : data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_combined_reference_set_9_islands.csv

[2/3] Applying epsilon-nondominated filtering...
  [fine  ]  epsilons: [0.001, 0.001, 0.001, 0.001, 0.001, 0.001]  →    35 policies  →  data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/MOMA_reference_set_fine.csv
  [coarse]  epsilons: [50, 50, 50, 50, 50, 0.1]  →     5 policies  →  data/temporary/MOMA_DATA/200k/J

## 1c. Testing the hypervolume

In [5]:
# ── Section 1c: Hypervolume Calculation & Plot ────────────────────────────────
from solvers.convergence.hypervolume import calculate_hypervolume_from_archives
from justice.util.visualizer import plot_hypervolume
import multiprocessing
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
base_name  = "MOMA_200000_ref2"
hv_dir     = f"{joint_path}/hypervolumes"
plots_dir  = f"{base_path}/plots"
os.makedirs(hv_dir,    exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# Islands 0–4 use the reference set from mm_intermediate_5.zip
# Islands 5–8 use the reference set from mm_intermediate_4.zip
island_ref_map = {
    0: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_5_islands.csv"},
    1: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_5_islands.csv"},
    2: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_5_islands.csv"},
    3: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_5_islands.csv"},
    4: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_5_islands.csv"},
    5: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_4_islands.csv"},
    6: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_4_islands.csv"},
    7: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_4_islands.csv"},
    8: {"ref_path": joint_path, "ref_file": "MOMA_reference_set_4_islands.csv"},
}

# ── STEP 1: Save per-run reference sets ──────────────────────────────────────
print("[1/3] Saving per-run reference sets...")
df_5.to_csv(f"{joint_path}/MOMA_reference_set_5_islands.csv", index=False)
df_4.to_csv(f"{joint_path}/MOMA_reference_set_4_islands.csv", index=False)
print(f"      5-island ref set : {len(df_5)} rows")
print(f"      4-island ref set : {len(df_4)} rows")

# ── STEP 2: Compute Hypervolume per island ────────────────────────────────────
print(f"\n[2/3] Computing hypervolumes for all 9 islands...")
hv_files = {}

with multiprocessing.Pool() as pool:
    for island_idx, ref_info in island_ref_map.items():
        filename = f"{base_name}_{island_idx}.tar.gz"
        print(f"      Island {island_idx}: {filename}  →  ref: {ref_info['ref_file']}")

        calculate_hypervolume_from_archives(
            list_of_objectives=list_of_objectives,
            direction_of_optimization=["max", "max", "max", "max", "max", "min"],
            input_data_path=joint_path,
            file_name=filename,
            output_data_path=hv_dir,
            saving=True,
            global_reference_set=True,
            global_reference_set_path=ref_info["ref_path"],
            global_reference_set_file=ref_info["ref_file"],
            pool=pool,
        )
        hv_files[island_idx] = f"hypervolumes/{base_name}_{island_idx}_hv.csv"

print(f"      Hypervolume files saved to: {hv_dir}")

# ── STEP 3: Plot Hypervolume ──────────────────────────────────────────────────
print("\n[3/3] Plotting hypervolume convergence...")

input_data_path_list = {
    "MOMA 200k (all 9 seeds)": [hv_files[i] for i in range(9)],
}

fig = plot_hypervolume(
    path_to_data=joint_path,
    path_to_output=plots_dir,
    input_data=input_data_path_list,
    yaxis_upper_limit=0.3,
    width=1000,
    height=800,
    fontsize=20,
    saving=True,
)

fig.show()
print(f"      Plot saved to: {plots_dir}")


[1/3] Saving per-run reference sets...
      5-island ref set : 264 rows
      4-island ref set : 1016 rows

[2/3] Computing hypervolumes for all 9 islands...
      Island 0: MOMA_200000_ref2_0.tar.gz  →  ref: MOMA_reference_set_5_islands.csv
Loading archives for MOMA_200000_ref2_0.tar.gz
Archives loaded
list_of_archives:  (3182, 6)
reference_set (264, 6)
type of reference_set <class 'numpy.ndarray'>
nfes: 
 [100, 10000, 100000, 110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 20000, 200000, 30000, 40000, 50000, 60000, 70000, 80000, 90000]
Computing hypervolume for  MOMA_200000_ref2_0.tar.gz
Time taken for Hypervolume Calculation: 3.434 seconds
data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/hypervolumes/MOMA_200000_ref2_0_hv.csv
      Island 1: MOMA_200000_ref2_1.tar.gz  →  ref: MOMA_reference_set_5_islands.csv
Loading archives for MOMA_200000_ref2_1.tar.gz
Archives loaded
list_of_archives:  (2552, 6)
reference_set (264, 6)
type of reference_set <class 'numpy.n

      Plot saved to: data/temporary/MOMA_DATA/200k/plots


In [8]:
# ── Section 1c: Hypervolume Calculation & Plot ────────────────────────────────
from solvers.convergence.hypervolume import calculate_hypervolume_from_archives
from justice.util.visualizer import plot_hypervolume
import multiprocessing
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
base_name  = "MOMA_200000_ref2"
hv_dir     = f"{joint_path}/hypervolumes"
plots_dir  = f"{base_path}/plots"
ref_file   = "MOMA_reference_set_fine.csv"   # fine-grained joint ref set for all islands
os.makedirs(hv_dir,    exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# ── STEP 1: Compute Hypervolume per island ────────────────────────────────────
print(f"[1/2] Computing hypervolumes for all 9 islands...")
print(f"      Reference set : {ref_file}\n")
hv_files = {}

with multiprocessing.Pool() as pool:
    for island_idx in range(9):
        filename = f"{base_name}_{island_idx}.tar.gz"
        print(f"      Island {island_idx}: {filename}")

        calculate_hypervolume_from_archives(
            list_of_objectives=list_of_objectives,
            direction_of_optimization=["max", "max", "max", "max", "max", "min"],
            input_data_path=joint_path,
            file_name=filename,
            output_data_path=hv_dir,
            saving=True,
            global_reference_set=True,
            global_reference_set_path=joint_path,
            global_reference_set_file=ref_file,
            pool=pool,
        )
        hv_files[island_idx] = f"hypervolumes/{base_name}_{island_idx}_hv.csv"

print(f"\n      Hypervolume files saved to: {hv_dir}")

# ── STEP 2: Plot Hypervolume ──────────────────────────────────────────────────
print("\n[2/2] Plotting hypervolume convergence...")

input_data_path_list = {
    "MOMA 200k (all 9 seeds)": [hv_files[i] for i in range(9)],
}

fig = plot_hypervolume(
    path_to_data=joint_path,
    path_to_output=plots_dir,
    input_data=input_data_path_list,
    yaxis_upper_limit=0.8,
    width=1000,
    height=800,
    fontsize=20,
    saving=True,
)

fig.show()
print(f"      Plot saved to: {plots_dir}")


[1/2] Computing hypervolumes for all 9 islands...
      Reference set : MOMA_reference_set_fine.csv

      Island 0: MOMA_200000_ref2_0.tar.gz
Loading archives for MOMA_200000_ref2_0.tar.gz
Archives loaded
list_of_archives:  (3182, 6)
reference_set (35, 6)
type of reference_set <class 'numpy.ndarray'>
nfes: 
 [100, 10000, 100000, 110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 20000, 200000, 30000, 40000, 50000, 60000, 70000, 80000, 90000]
Computing hypervolume for  MOMA_200000_ref2_0.tar.gz
Time taken for Hypervolume Calculation: 3.209 seconds
data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/hypervolumes/MOMA_200000_ref2_0_hv.csv
      Island 1: MOMA_200000_ref2_1.tar.gz
Loading archives for MOMA_200000_ref2_1.tar.gz
Archives loaded
list_of_archives:  (2552, 6)
reference_set (35, 6)
type of reference_set <class 'numpy.ndarray'>
nfes: 
 [100, 10000, 100000, 110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 20000, 200000, 30000, 40000, 50000

      Plot saved to: data/temporary/MOMA_DATA/200k/plots


# 2. Pareto Nash

In [ ]:
# ── Section 2: Pareto-Nash Filtering ─────────────────────────────────────────
import pandas as pd
import os
from justice.util.pareto_nash_search import is_pareto_dominated
# ── CONFIG ────────────────────────────────────────────────────────────────────
base_path = "data/temporary/MOMA_DATA/200k"
processed_path = f"{base_path}/PROCESSED_DATA"
joint_path = f"{base_path}/JOINT_OPTIMIZATION"
os.makedirs(processed_path, exist_ok=True)

input_file  = f"{joint_path}/COMBINED_MOMA_epsilon_nondominated_set.csv"
output_file = f"{processed_path}/pareto_nash_profiles.csv"

agent_objectives = {
    "agent_1": ["macro_welfare_R5ASIA", "fraction_above_threshold"],
    "agent_2": ["macro_welfare_R5LAM",  "fraction_above_threshold"],
    "agent_3": ["macro_welfare_R5MAF",  "fraction_above_threshold"],
    "agent_4": ["macro_welfare_R5OECD", "fraction_above_threshold"],
    "agent_5": ["macro_welfare_R5REF",  "fraction_above_threshold"],
}
minimize_objectives = ["fraction_above_threshold"]

# ── LOAD ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(input_file).reset_index(drop=True)
print(f"Loaded {len(df)} candidate solutions from: {input_file}\n")


# ── PARETO-NASH FILTERING ─────────────────────────────────────────────────────
print("Checking Pareto-Nash conditions for each solution...\n")
pareto_nash_indices = []

for solution_idx in df.index:
    is_pareto_nash = all(
        not is_pareto_dominated(
            solution_idx, agent_name, df,
            agent_objectives, minimize_objectives,
        )
        for agent_name in agent_objectives
    )

    if is_pareto_nash:
        pareto_nash_indices.append(solution_idx)
        print(f"  Solution {solution_idx:>3} is PARETO-NASH ✓")

# ── RESULTS ───────────────────────────────────────────────────────────────────
pareto_nash_df = df.loc[pareto_nash_indices].copy().reset_index(drop=True)

print(f"\nFound {len(pareto_nash_df)} Pareto-Nash solutions out of {len(df)} candidates.")
print(f"\nObjective values for Pareto-Nash set:")
print(pareto_nash_df[list(agent_objectives["agent_1"][:-1]) +
                     [f"macro_welfare_{r}" for r in ["R5LAM","R5MAF","R5OECD","R5REF"]] +
                     ["fraction_above_threshold"]].to_string(index=True))

# pareto_nash_df.to_csv(output_file, index=True, index_label="profile_index")
# print(f"\nSaved to: {output_file}")


Loaded 5 candidate solutions from: data/temporary/MOMA_DATA/200k/JOINT_OPTIMIZATION/COMBINED_MOMA_epsilon_nondominated_set.csv

Checking Pareto-Nash conditions for each solution...

  Solution   4 is PARETO-NASH ✓

Found 1 Pareto-Nash solutions out of 5 candidates.

Objective values for Pareto-Nash set:
   macro_welfare_R5ASIA  macro_welfare_R5LAM  macro_welfare_R5MAF  macro_welfare_R5OECD  macro_welfare_R5REF  fraction_above_threshold
0             1707.0187          3027.130867          2137.421176             3372.8681          2145.106776                      0.42


### Section 2b: Payoff Table Generation & Pareto-Nash Profile Extraction

This section computes the **Pareto-Nash equilibrium profiles** for the 5-agent climate policy game. It proceeds in three steps:

**Step 1 — Benchmark.** A quick sequential timing test runs a small sample of profiles (default: 10) to estimate how long the full computation will take. This step is optional and can be commented out after the first run.

**Step 2 — Payoff table construction.** All 5⁵ = 3,125 possible action combinations across the 5 macro-regional agents are enumerated and simulated through the full JUSTICE model. Each simulation runs the complete climate-economy time loop across all FaIR ensemble members, computing regional welfare and the fraction of ensemble members exceeding 2°C warming by the target year. Simulations are distributed across available CPU cores using multiprocessing to reduce wall time. Results are written incrementally to `payoff_table_5x5x5x5x5.csv` in the processed data folder.

> ⚠️ **This is the most computationally expensive step in the analysis.** Expect **30–90 minutes** of wall time depending on hardware and the number of workers. A skip guard prevents re-running if the output file already exists — do not delete this file unless the underlying policies or model configuration have changed.

**Step 3 — Pareto-Nash extraction.** The payoff table is filtered to retain only **Pareto-Nash equilibrium profiles** — action combinations where no agent can unilaterally switch to a strategy that improves their own welfare *and* the climate outcome simultaneously. For each agent, best responses are identified by fixing all other agents' actions and finding the Pareto-nondominated strategies (trading off welfare against fraction above 2°C). A profile is classified as Pareto-Nash only if every agent is simultaneously on their own Pareto frontier. The resulting profiles are saved to `pareto_nash_profiles.csv` and passed to Section 3 for full simulation.

In [ ]:
# ── Section 2: Generate Payoff Table & Extract Pareto-Nash Profiles ───────────
from justice.util.pareto_nash_search import (
    benchmark_profiles,
    run_5pow5_payoff_table,
    pareto_nash_set,
)
import multiprocessing as mp
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
policy_csv_path = f"{joint_path}/COMBINED_MOMA_epsilon_nondominated_set.csv"
config_path     = "analysis/momadps_config.json"
payoff_csv_path = f"{processed_path}/payoff_table_5x5x5x5x5.csv"
nash_out_path   = f"{processed_path}/pareto_nash_profiles.csv"
n_workers       = max(1, mp.cpu_count() - 1)   # tune down if memory is tight
chunksize       = 8

# ── STEP 1: Benchmark (optional — comment out after first run) ────────────────
print("[0/3] Benchmarking to estimate runtime...")
benchmark_profiles(
    policy_csv_path=policy_csv_path,
    config_path=config_path,
    scenario=2,
    mapping_base_path="data/input",
    n_samples=10,
    seed=0,
)

# ── STEP 2: Run full 5^5 payoff table ────────────────────────────────────────
# ⚠ This is the longest step — expect 30–90 min depending on n_workers and hardware
# Skip if payoff_csv_path already exists

if os.path.exists(payoff_csv_path):
    print(f"\n[1/3] Payoff table already exists, skipping: {payoff_csv_path}")
else:
    print(f"\n[1/3] Running 5^5 = 3125 profiles with {n_workers} workers...")
    run_5pow5_payoff_table(
        policy_csv_path=policy_csv_path,
        config_path=config_path,
        out_csv_path=payoff_csv_path,
        scenario=2,
        mapping_base_path="data/input",
        n_workers=n_workers,
        chunksize=chunksize,
    )
    print(f"      Saved to: {payoff_csv_path}")

# ── STEP 3: Extract Pareto-Nash profiles ──────────────────────────────────────
print(f"\n[2/3] Extracting Pareto-Nash profiles from payoff table...")
pn_df = pareto_nash_set(
    payoff_csv_path=payoff_csv_path,
    minimize_fraction=True,
    fraction_col="fraction_above_threshold",
    welfare_prefix="welfare_",
    action_prefix="a",
    n_agents=None,   # inferred from columns
)

pn_df.to_csv(nash_out_path, index=False)

print(f"      Found {len(pn_df)} Pareto-Nash profiles out of 3125.")
print(f"      Saved to: {nash_out_path}")
print(f"\n      Preview:")
cols = [c for c in pn_df.columns if c.startswith("a") or
        c.startswith("welfare_")] + ["fraction_above_threshold"]
print(pn_df[cols].to_string(index=True))


[0/3] Benchmarking to estimate runtime...
Benchmark: 10 samples in 4.787s
Time/profile: 0.4787s
Estimated time for 5^5=3125 profiles: 24.93 minutes

[1/3] Running 5^5 = 3125 profiles with 9 workers...
100/3125 done | 5.65 проф/s | ETA 8.9 min
200/3125 done | 7.38 проф/s | ETA 6.6 min
300/3125 done | 7.35 проф/s | ETA 6.4 min
400/3125 done | 8.25 проф/s | ETA 5.5 min
500/3125 done | 8.68 проф/s | ETA 5.0 min
600/3125 done | 8.74 проф/s | ETA 4.8 min
700/3125 done | 9.10 проф/s | ETA 4.4 min
800/3125 done | 9.32 проф/s | ETA 4.2 min
900/3125 done | 9.33 проф/s | ETA 4.0 min
1000/3125 done | 9.56 проф/s | ETA 3.7 min
1100/3125 done | 9.68 проф/s | ETA 3.5 min
1200/3125 done | 9.73 проф/s | ETA 3.3 min
1300/3125 done | 9.93 проф/s | ETA 3.1 min
1400/3125 done | 9.81 проф/s | ETA 2.9 min
1500/3125 done | 9.85 проф/s | ETA 2.8 min
1600/3125 done | 9.84 проф/s | ETA 2.6 min
1700/3125 done | 9.83 проф/s | ETA 2.4 min
1800/3125 done | 9.93 проф/s | ETA 2.2 min
1900/3125 done | 9.86 проф/s | ETA

### 2b Selecting Pareto-Nash profiles for simulation

In [2]:
# ── Section 2b: Select Focal Pareto-Nash Profiles ────────────────────────────
import pandas as pd
import numpy as np

base_path  = "data/temporary/MOMA_DATA/200k"
processed_path = f"{base_path}/PROCESSED_DATA"
nash_out_path   = f"{processed_path}/pareto_nash_profiles.csv"

# ── Load Pareto-Nash set ──────────────────────────────────────────────────────
pn_df = pd.read_csv(nash_out_path)
action_cols  = [c for c in pn_df.columns if c.startswith("a") and c[1:].isdigit()]
welfare_cols = [c for c in pn_df.columns if c.startswith("welfare_")]
n_agents     = len(action_cols)

# ── PROFILE 1: Joint optimization candidate ───────────────────────────────────
# In the joint Pareto front, all agents share the same policy row (uniform action).
# We identify this as the profile where a0 == a1 == ... == a4.
joint_mask     = pn_df[action_cols].nunique(axis=1) == 1
joint_profiles = pn_df[joint_mask]

if len(joint_profiles) == 0:
    raise RuntimeError(
        "No uniform-action profile found in Pareto-Nash set. "
        "Check if the joint optimization candidate was correctly included."
    )
elif len(joint_profiles) > 1:
    # If multiple uniform-action profiles exist, pick the one with
    # lowest fraction_above_threshold (most balanced mitigation)
    joint_profile = joint_profiles.loc[
        joint_profiles["fraction_above_threshold"].idxmin()
    ]
    print(f"  Note: {len(joint_profiles)} uniform-action profiles found; "
          f"selected the one with lowest fraction_above_threshold.")
else:
    joint_profile = joint_profiles.iloc[0]

joint_idx = int(joint_profile.name)

# ── PROFILE 2: Best temperature outcome ───────────────────────────────────────
best_temp_idx  = int(pn_df["fraction_above_threshold"].idxmin())
best_temp_profile = pn_df.loc[best_temp_idx]

# ── PROFILE 3: Worst temperature outcome ──────────────────────────────────────
worst_temp_idx = int(pn_df["fraction_above_threshold"].idxmax())
worst_temp_profile = pn_df.loc[worst_temp_idx]

# ── Summary ───────────────────────────────────────────────────────────────────
selected_indices = [joint_idx, best_temp_idx, worst_temp_idx]
labels = {
    joint_idx:      "Joint optimization candidate (balanced)",
    best_temp_idx:  "Best temperature outcome",
    worst_temp_idx: "Worst temperature outcome",
}

print(f"\nSelected Pareto-Nash profiles ({len(selected_indices)} of {len(pn_df)}):\n")
display_cols = action_cols + welfare_cols + ["fraction_above_threshold"]

for idx, label in labels.items():
    row = pn_df.loc[idx]
    actions  = row[action_cols].tolist()
    welfares = [f"{row[c]:.4f}" for c in welfare_cols]
    frac     = row["fraction_above_threshold"]
    print(f"  [{idx:>3}]  {label}")
    print(f"         Actions  : {actions}")
    print(f"         Welfares : {welfares}")
    print(f"         Frac>2°C : {frac:.4f}\n")

# ── Save selection metadata ───────────────────────────────────────────────────
selection_meta = {
    "joint_candidate_index":  joint_idx,
    "best_temperature_index": best_temp_idx,
    "worst_temperature_index": worst_temp_idx,
}

import json
meta_path = f"{processed_path}/focal_profile_indices.json"
with open(meta_path, "w") as f:
    json.dump(selection_meta, f, indent=2)

print(f"Focal profile indices saved to: {meta_path}")



Selected Pareto-Nash profiles (3 of 88):

  [ 86]  Joint optimization candidate (balanced)
         Actions  : [4, 4, 4, 4, 4]
         Welfares : ['1707.0187', '3027.1309', '2137.4212', '3372.8681', '2145.1068']
         Frac>2°C : 0.4200

  [  9]  Best temperature outcome
         Actions  : [0, 1, 4, 3, 2]
         Welfares : ['1710.8636', '3032.3700', '2159.9756', '3368.8039', '2119.1604']
         Frac>2°C : 0.3800

  [ 52]  Worst temperature outcome
         Actions  : [3, 3, 3, 2, 0]
         Welfares : ['1682.5163', '3001.4970', '2132.4574', '3442.0717', '2182.3716']
         Frac>2°C : 0.6000

Focal profile indices saved to: data/temporary/MOMA_DATA/200k/PROCESSED_DATA/focal_profile_indices.json


## 3. Simulating Pareto-Nash policies


> ⚠️ **This is also computationally expensive and will require sufficient memory.** 


In [ ]:
# ── Section 3: Simulate Focal Pareto-Nash Profiles ───────────────────────────
import json
from justice.util.pareto_nash_simulate import simulate_nash_profiles_by_row_index

with open(f"{processed_path}/focal_profile_indices.json") as f:
    focal = json.load(f)

selected_profile_indices = list(focal.values())   # [joint, best_temp, worst_temp]
print(f"Simulating profiles: {selected_profile_indices}")

simulate_nash_profiles_by_row_index(
    nash_profiles_csv        = f"{processed_path}/pareto_nash_profiles.csv",
    policy_5row_csv          = f"{joint_path}/COMBINED_MOMA_epsilon_nondominated_set.csv",
    config_path              = "analysis/momadps_config.json",
    out_dir                  = f"{processed_path}/simulations",
    selected_profile_indices = selected_profile_indices,
    scenario                 = 2,
    mapping_base_path        = "data/input",
)


## 4. Single Agent Decentralized Optimization

In [1]:
# ── Section 4a: Extract Single-Agent Borg Archives ───────────────────────────
# 4 islands per agent (5-island runs still in progress — re-run this cell when ready)

import os

base_path  = "data/temporary/MOMA_DATA/200k"
single_agent_path = f"{base_path}/SINGLE_AGENT_TESTS"   # adjust if different
policies  = [9, 52, 86]
n_agents  = 5

for policy_idx in policies:
    for agent_idx in range(n_agents):
        archive = (
            f"{single_agent_path}/policy_{policy_idx}/"
            f"mm_intermediate_agent_{agent_idx}.zip"
        )
        base_name = f"SINGLE_AGENT_policy{policy_idx}_agent{agent_idx}"

        if not os.path.exists(archive):
            print(f"  [SKIP] Not found: {archive}")
            continue

        print(f"\n  Extracting policy={policy_idx}, agent={agent_idx}...")
        !python justice/util/borg_archive_processor.py \
            --archive "{archive}" \
            --base-name "{base_name}" \
            --step 10000 \
            --island-offset 0



  Extracting policy=9, agent=0...
Archive        : data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/mm_intermediate_agent_0.zip
Islands found  : 4
Island offset  : 0  → indices 0..3
Step filter    : every 10000
Output dir     : data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9

[OK] data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/SINGLE_AGENT_policy9_agent0_0.tar.gz  (source island: mm_0)
[OK] data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/SINGLE_AGENT_policy9_agent0_1.tar.gz  (source island: mm_1)
[OK] data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/SINGLE_AGENT_policy9_agent0_2.tar.gz  (source island: mm_2)
[OK] data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/SINGLE_AGENT_policy9_agent0_3.tar.gz  (source island: mm_3)

Done.

  Extracting policy=9, agent=1...
Archive        : data/temporary/MOMA_DATA/200k/SINGLE_AGENT_TESTS/policy_9/mm_intermediate_agent_1.zip
Islands found  : 4
Island offset  : 0  → indices 0..3
Step filter 

### 4b. Extract and build reference set for single agents

In [5]:
# ── Section 4b: Extract & Build Per-Agent Reference Sets ─────────────────────
from solvers.convergence.pareto import eps_sort
import numpy as np
import pandas as pd
import zipfile
import tarfile
import io
import os
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
single_agent_path = f"{base_path}/SINGLE_AGENT_TESTS"

policies      = [9, 52, 86]
n_agents      = 5        # zip files: mm_intermediate_agent_0.zip .. agent_4.zip
fine_epsilons = [0.001, 0.001]


# ── HELPER: extract all 200000.csv seeds from a zip ──────────────────────────
def extract_final_csv_from_zip(zip_path: str) -> pd.DataFrame:
    dfs = []
    with zipfile.ZipFile(zip_path, "r") as outer_zip:
        tar_names = [n for n in outer_zip.namelist() if n.endswith(".tar.gz")]
        print(f"        Found {len(tar_names)} tar.gz in {os.path.basename(zip_path)}")

        for tar_name in tar_names:
            with outer_zip.open(tar_name) as tar_bytes:
                with tarfile.open(fileobj=io.BytesIO(tar_bytes.read()), mode="r:gz") as tar:
                    csv_members = [m for m in tar.getmembers()
                                   if m.name.endswith("200000.csv")]
                    if not csv_members:
                        print(f"        [WARN] No 200000.csv in {tar_name}, skipping.")
                        continue
                    f = tar.extractfile(csv_members[0])
                    df = pd.read_csv(f)
                    dfs.append(df)
                    print(f"        Loaded {len(df):>6} rows ← {tar_name}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def infer_welfare_col(df: pd.DataFrame) -> str:
    welfare_cols = [c for c in df.columns if c.startswith("macro_welfare_")]
    if len(welfare_cols) != 1:
        raise ValueError(f"Expected exactly 1 macro_welfare_* column, found: {welfare_cols}")
    return welfare_cols[0]


# ── MAIN LOOP: policy × agent ─────────────────────────────────────────────────
print("Building per-agent reference sets...\n")
print(f"{'Policy':>8}  {'Agent':>5}  {'Welfare col':<25}  {'Raw rows':>9}  {'Ref set':>8}")
print("─" * 80)

for policy_idx in policies:
    policy_dir = f"{single_agent_path}/policy_{policy_idx}"
    out_dir    = f"{policy_dir}/reference_sets"
    os.makedirs(out_dir, exist_ok=True)

    for agent_idx in range(n_agents):     # 0-indexed: 0..4
        archive  = f"{policy_dir}/mm_intermediate_agent_{agent_idx}.zip"
        out_path = f"{out_dir}/reference_set_agent_{agent_idx}.csv"

        if not os.path.exists(archive):
            print(f"  policy_{policy_idx} / agent_{agent_idx}: [SKIP] archive not found")
            continue

        # ── Extract & concatenate all seeds ──────────────────────────────────
        df_raw = extract_final_csv_from_zip(archive)
        if df_raw.empty:
            print(f"  policy_{policy_idx} / agent_{agent_idx}: [SKIP] no data extracted")
            continue

        # ── Infer welfare column from CSV ─────────────────────────────────────
        welfare_col = infer_welfare_col(df_raw)

        # ── Epsilon-nondominated filtering ────────────────────────────────────
        table = df_raw.reset_index()[
            ["index", welfare_col, "fraction_above_threshold"]
        ].to_numpy()

        tagalongs  = eps_sort(
            table,
            objectives=[1, 2],      # ← was [1,2,3] — only 2 objectives here
            epsilons=fine_epsilons,
            maximize=[1],           # welfare → maximize; fraction → minimize
        )
        tagalongs  = np.array(tagalongs)
        pareto_idx = np.unique(tagalongs[:, 0].astype(int))
        ref_df     = df_raw.loc[pareto_idx].copy()


        ref_df.to_csv(out_path, index=False)
        print(f"  {policy_idx:>8}  {agent_idx:>5}  {welfare_col:<25}  "
              f"{len(df_raw):>9}  {len(ref_df):>8}")

print("\nDone.")


Building per-agent reference sets...

  Policy  Agent  Welfare col                 Raw rows   Ref set
────────────────────────────────────────────────────────────────────────────────
        Found 1 tar.gz in mm_intermediate_agent_0.zip
        Loaded     25 rows ← SingleAgent_0_200000_4444.tar.gz
         9      0  macro_welfare_R5ASIA              25         1
        Found 1 tar.gz in mm_intermediate_agent_1.zip
        Loaded      2 rows ← SingleAgent_1_200000_4444.tar.gz
         9      1  macro_welfare_R5LAM                2         1
        Found 1 tar.gz in mm_intermediate_agent_2.zip
        Loaded     10 rows ← SingleAgent_2_200000_4444.tar.gz
         9      2  macro_welfare_R5MAF               10         1
        Found 1 tar.gz in mm_intermediate_agent_3.zip
        Loaded      1 rows ← SingleAgent_3_200000_4444.tar.gz
         9      3  macro_welfare_R5OECD               1         1
        Found 1 tar.gz in mm_intermediate_agent_4.zip
        Loaded      1 rows ← Single

### 4c. Check if Single-Agent Optimization Dominates Joint Solution

In [7]:
# ── Section 4c: Check if Single-Agent Optimization Dominates Joint Solution ───
import pandas as pd
import numpy as np
import os
from justice.util.pareto_nash_search import is_pareto_dominated

# ── CONFIG ────────────────────────────────────────────────────────────────────
focal_policies      = [9, 52, 86]
n_agents            = 5   # 0-indexed: 0..4
minimize_objectives = {"fraction_above_threshold"}

base_path  = "data/temporary/MOMA_DATA/200k"
processed_path = f"{base_path}/PROCESSED_DATA"
nash_out_path   = f"{processed_path}/pareto_nash_profiles.csv"
single_agent_path = f"{base_path}/SINGLE_AGENT_TESTS"

# ── LOAD Pareto-Nash profiles ─────────────────────────────────────────────────
pn_df = pd.read_csv(nash_out_path)

print("=" * 72)
print("DOMINATION CHECK: Single-Agent Opt vs. Joint Opt (Pareto-Nash profile)")
print("=" * 72)

for policy_idx in focal_policies:

    if policy_idx not in pn_df.index:
        print(f"\n[WARN] Policy index {policy_idx} not found in pareto_nash_profiles.csv")
        continue

    joint_row  = pn_df.loc[policy_idx]
    joint_frac = float(joint_row["fraction_above_threshold"])

    print(f"\n{'─'*72}")
    print(f"  POLICY INDEX : {policy_idx}")
    print(f"  Actions      : {[int(joint_row[f'a{i}']) for i in range(n_agents)]}")
    print(f"  Frac > 2°C   : {joint_frac:.6f}")
    print(f"{'─'*72}")

    for agent_idx in range(n_agents):
        ref_path = (
            f"{single_agent_path}/policy_{policy_idx}/"
            f"reference_sets/reference_set_agent_{agent_idx}.csv"
        )

        if not os.path.exists(ref_path):
            print(f"  Agent {agent_idx}: [SKIP] reference set not found")
            continue

        ref_df      = pd.read_csv(ref_path)
        welfare_col = [c for c in ref_df.columns if c.startswith("macro_welfare_")][0]

        # Joint solution values for this agent
        joint_welfare = float(joint_row[f"welfare_{agent_idx}"])

        print(f"\n  Agent {agent_idx}  [{welfare_col}]")
        print(f"    Joint solution  →  welfare: {joint_welfare:.6f}  "
              f"|  frac: {joint_frac:.6f}")

        # ── Build unified df: rename welfare col so is_pareto_dominated works ─
        # Both joint and ref set must share the same column names
        ref_df_cmp = ref_df[[welfare_col, "fraction_above_threshold"]].copy()
        ref_df_cmp = ref_df_cmp.rename(columns={welfare_col: "welfare"})

        joint_entry = pd.DataFrame([{
            "welfare":                  joint_welfare,
            "fraction_above_threshold": joint_frac,
        }])

        combined       = pd.concat([ref_df_cmp, joint_entry], ignore_index=True)
        joint_comb_idx = combined.index[-1]   # joint solution is the last row

        # ── Reuse is_pareto_dominated ─────────────────────────────────────────
        agent_objectives = {
            "agent": ["welfare", "fraction_above_threshold"]
        }

        dominated = is_pareto_dominated(
            solution_idx      = joint_comb_idx,
            agent_name        = "agent",
            df                = combined,
            agent_objectives  = agent_objectives,
            minimize_objectives = minimize_objectives,
        )

        if dominated:
            # Find the dominating solution(s) for informative printout
            for other_idx in combined.index:
                if other_idx == joint_comb_idx:
                    continue
                other = combined.loc[other_idx]
                w_ok  = other["welfare"] >= joint_welfare
                f_ok  = other["fraction_above_threshold"] <= joint_frac
                w_better = other["welfare"] > joint_welfare
                f_better = other["fraction_above_threshold"] < joint_frac
                if w_ok and f_ok and (w_better or f_better):
                    print(f"    ⚠  DOMINATED by single-agent solution:")
                    print(f"       welfare: {other['welfare']:.6f}  "
                          f"|  frac: {other['fraction_above_threshold']:.6f}")
                    break
        else:
            best_w    = ref_df_cmp["welfare"].max()
            best_frac = ref_df_cmp["fraction_above_threshold"].min()
            print(f"    ✓  NOT dominated")
            print(f"       Best single-agent welfare  : {best_w:.6f}  "
                  f"({'better' if best_w > joint_welfare else 'worse or equal'} than joint)")
            print(f"       Best single-agent frac>2°C : {best_frac:.6f}  "
                  f"({'better' if best_frac < joint_frac else 'worse or equal'} than joint)")

print(f"\n{'='*72}")
print("Done.")


DOMINATION CHECK: Single-Agent Opt vs. Joint Opt (Pareto-Nash profile)

────────────────────────────────────────────────────────────────────────
  POLICY INDEX : 9
  Actions      : [0, 1, 4, 3, 2]
  Frac > 2°C   : 0.380000
────────────────────────────────────────────────────────────────────────

  Agent 0  [macro_welfare_R5ASIA]
    Joint solution  →  welfare: 1710.863627  |  frac: 0.380000
    ✓  NOT dominated
       Best single-agent welfare  : 1709.290606  (worse or equal than joint)
       Best single-agent frac>2°C : 0.380000  (worse or equal than joint)

  Agent 1  [macro_welfare_R5LAM]
    Joint solution  →  welfare: 3032.370025  |  frac: 0.380000
    ✓  NOT dominated
       Best single-agent welfare  : 2973.697602  (worse or equal than joint)
       Best single-agent frac>2°C : 0.360000  (better than joint)

  Agent 2  [macro_welfare_R5MAF]
    Joint solution  →  welfare: 2159.975633  |  frac: 0.380000
    ✓  NOT dominated
       Best single-agent welfare  : 2149.263147  (worse